# Declino italiano: dashboard API

Questo notebook costruisce un cruscotto comparabile Italia-Germania-Francia-Spagna-UE usando API ufficiali.
Le fonti sono World Bank (WDI, WGI, Doing Business storico) ed Eurostat (`nama_10_lp_ulc`).

L'obiettivo non e produrre uno slogan, ma rendere riproducibile la diagnosi: per ogni indicatore guardiamo trend dell'Italia e gap rispetto ai principali peer europei.

In [ ]:
from pathlib import Path
import sys

# Il notebook puo essere eseguito dalla root del repo oppure dalla cartella notebooks.
ROOT = Path.cwd()
if not (ROOT / "analisi").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Image, display

from analisi._comune.indicatori_api import run_decline_analysis, plot_indicator, OUTDIR

# Metti True per riscaricare le API; False usa i CSV gia salvati se presenti.
REFRESH_API = False

result = run_decline_analysis(refresh=REFRESH_API)
panel = result["panel"]
summary = result["summary"]

print("Output:", OUTDIR)
summary.head(10)

## Come leggere la tabella

`bad_when` e la direzione interpretativa: `low` significa che valori piu bassi sono peggiori, `high` che valori piu alti sono peggiori.  
`worsened_since_baseline` segnala peggioramento rispetto al primo dato disponibile dal 2000.  
`italy_worse_than_peer_avg` segnala Italia peggiore della media Germania-Francia-Spagna-UE nell'ultimo anno disponibile.

In [ ]:
cols = [
    "theme", "indicator_label", "unit", "latest_year", "latest_value_italy",
    "change_abs_since_baseline", "gap_vs_peer_mean",
    "worsened_since_baseline", "italy_worse_than_peer_avg", "source"
]
critical = summary[
    summary["worsened_since_baseline"] | summary["italy_worse_than_peer_avg"]
].sort_values(["theme", "italy_worse_than_peer_avg", "worsened_since_baseline"], ascending=[True, False, False])
critical[cols]

## Quadro per tema

Qui contiamo quanti indicatori, dentro ogni tema, segnalano peggioramento storico o svantaggio rispetto ai peer.

In [ ]:
by_theme = (
    summary.assign(criticita=summary["worsened_since_baseline"] | summary["italy_worse_than_peer_avg"])
    .groupby("theme", as_index=False)
    .agg(indicatori=("indicator_id", "nunique"), indicatori_critici=("criticita", "sum"))
    .sort_values("indicatori_critici", ascending=False)
)
by_theme

## Grafici chiave

Le linee mostrano Italia e peer. L'Italia e evidenziata con tratto piu spesso.

In [ ]:
key_indicators = [
    "ESTAT_RLPR_HW_I15",      # produttivita per ora lavorata
    "SP.DYN.TFRT.IN",        # fertilita
    "SP.POP.65UP.TO.ZS",     # invecchiamento
    "GB.XPD.RSDV.GD.ZS",     # R&S
    "PAY.TAX.TM",            # ore per adempimenti fiscali
    "GC.XPN.INTP.RV.ZS",     # interessi sul debito
    "NE.GDI.FTOT.ZS",        # investimenti fissi
    "GOV_WGI_GE.EST",        # efficacia del governo
]

for indicator_id in key_indicators:
    path = plot_indicator(panel, indicator_id)
    display(Image(filename=str(path)))

## Fonti salvate

La pipeline salva anche `declino_italia_sources.json`, con URL API e nomi degli indicatori. Questo rende tracciabile ogni serie usata nel notebook.

In [ ]:
sources = pd.read_json(OUTDIR / "declino_italia_sources.json")
sources.head(20)